In [2]:
import numpy as np
import itertools

In [3]:
def init_cd(n: int)-> np.ndarray:
    try:
        """Devuelve un array con valores −1 en las posiciones {0, 1, ..., n-1}."""
        if n < 0:
            return None
        return -1 * np.ones(n).astype(int)
    except:
        print("Error en init_cd")

In [4]:
print(init_cd(5))

[-1 -1 -1 -1 -1]


In [5]:
def union(rep_1: int, rep_2: int, p_cd: np.ndarray)-> int:
    try:
        """Devuelve el representante del conjunto obtenido como la union por rangos de los representados por los  ́ındices
            rep_1, rep_2 en el CD almacenado en el array p_cd."""
        if rep_1 > len(p_cd) or rep_2 > len(p_cd):
            return None
        if p_cd[rep_2] < p_cd[rep_1]:
            p_cd[rep_1] = rep_2; return rep_2
        elif p_cd[rep_2] > p_cd[rep_1]:
            p_cd[rep_2] = rep_1; return rep_1
        else:
            p_cd[rep_2] = rep_1; p_cd[rep_1] -= 1; return rep_1
    except:
        print("Error en union")

In [6]:
u=init_cd(5)
print(union(1,2,u))
print(u)

1
[-1 -2  1 -1 -1]


In [7]:
def find(ind: int, p_cd: np.ndarray)-> int:
    try:
        """Devuelve el representante del ́ındice ind en el CD almacenado en p_cd realizando compresión de caminos."""
        if ind > len(p_cd) or ind < 0:
            return None
        z = ind
        while p_cd[z] >= 0:
            z = p_cd[z]
        while p_cd[ind] >= 0:
            y = p_cd[ind]
            p_cd[ind] = z
            ind = y
        return z
    except:
        print("Error en find")

In [8]:
u=init_cd(5)
print(union(1,2,u))
union(find(2,u),find(3,u),u)
print(u)
print(find(3,u))

1
[-1 -2  1  1 -1]
1


In [9]:
def cd_2_dict(p_cd: np.ndarray)-> dict:
    try:
        """Recibe un CD en el array p_cd y devuelve un diccionario cuyas claves sean los representantes de los subconjuntos
            del CD y donde el valor de la clave u del dict sea una lista con los miembros del subconjunto """
        d = {}
        for i in range(0,len(p_cd)):
            if(p_cd[i]<0):
                l=[]
                for j in range (0,len(p_cd)):
                    m=find(j, p_cd)
                    if(m==i):
                        l.append(j)
                d[i]=l
        return d
    except:
        print("Error en cd_2_dict")

In [10]:
u=init_cd(5)
print(union(1,2,u))
union(find(2,u),find(3,u),u)
union(find(0,u),find(4,u),u)
print(u)
print(cd_2_dict(u))

1
[-2 -2  1  1  0]
{0: [0, 4], 1: [1, 2, 3]}


In [11]:
def ccs(n: int, l: list)-> dict:
    try:
        """Devuelve las componentes conexas de un tal grafo mediante un diccionario."""
        cd=init_cd(n)
        for i in l: 
            p1=find(i[0],cd)
            p2=find(i[1],cd)
            j=union(p1,p2,cd)
        return cd_2_dict(cd)
    except:
        print("Error en ccs")

In [12]:
g = [(0, 12), (10, 0), (7, 12), (1, 9), (3, 8), (3, 9), (11, 6), (2, 5), (4, 5), (0, 2)]
print(ccs(13,g))

{0: [0, 2, 4, 5, 7, 10, 12], 3: [1, 3, 8, 9], 11: [6, 11]}


In [13]:
def dist_matrix(n_nodes: int, w_max=10)-> np.ndarray:
    try:
        """Genera la matriz de distancias de un grafo con n_nodes nodos, valores enteros con un maximo w_max"""
        m_1=[]
        m_1 = np.random.randint(0, w_max, (n_nodes, n_nodes))
        m_1 = (m_1 + m_1.T) // 2
        m_1 = m_1 - np.diag( np.diag(m_1))
        return m_1
    except:
        print("Error en dist_matrix")

In [14]:
print(dist_matrix(4))

[[0 7 6 3]
 [7 0 3 1]
 [6 3 0 4]
 [3 1 4 0]]


In [77]:
def greedy_tsp(dist_m: np.ndarray, node_ini=0)-> list:
    try:
        """Recibe una matriz de distancias y un nodo inicial y devuelve un circuito codiciosos como una lista con valores entre
            0 y el n ́umero de nodos menos 1."""
        num_nodes = dist_m.shape[0]
        circuit = [node_ini]
        while len(circuit) < num_nodes:
            current_city = circuit[-1]
            options = list(np.argsort(dist_m[current_city]))
            for node in options:
                if node not in circuit:
                    circuit.append(node)
                    break
        return circuit + [node_ini]
    except:
        print("Error en greedy_tspgreedy_tsp")


In [78]:
print(greedy_tsp(dist_matrix(4,5),))

[0, 1, 2, 3, 0]


In [79]:
def len_circuit(circuit: list, dist_m: np.ndarray)-> int:
    try:
        """Recibe un circuito y una matriz de distancias y devuelve la longitud de dicho circuito."""
        i=0
        dist=0
        while i < (len(circuit) - 1):
            dist += dist_m[circuit[i]][circuit[i+1]]
            i += 1
        return dist
    except:
        print("Error en len_circuit")

In [80]:
m = dist_matrix(4, 5)
c = greedy_tsp(m, 0)
print(m)
print(c)
print(len_circuit(c, m))

[[0 1 2 1]
 [1 0 2 3]
 [2 2 0 0]
 [1 3 0 0]]
[0, 1, 2, 3, 0]
4


In [81]:
def repeated_greedy_tsp(dist_m: np.ndarray)-> list:
    try:
        """Llama a greedy_tsp a partir de todos los nodos del grafo y devuelve el circuito con la menor longitud."""
        best_circuit=greedy_tsp(dist_m,0)
        current_n=len_circuit(best_circuit,dist_m)
        #print(current_n)
        #print(best_circuit)
        for i in range(1,len(dist_m)):
            new_circuit=greedy_tsp(dist_m,i)
            new_n=len_circuit(new_circuit,dist_m)
            #print(new_n)
            #print(new_circuit)
            if(new_n<current_n):
                best_circuit=new_circuit
                current_n=new_n

        return best_circuit
    except:
        print("Error en repeated_greedy_tsp")

In [87]:
m = dist_matrix(200)
c = repeated_greedy_tsp(m)
#print(m)
print(c)
#print(len_circuit(c, m))

[131, 137, 1, 67, 42, 35, 117, 78, 196, 0, 181, 157, 158, 62, 161, 123, 38, 162, 170, 105, 126, 185, 86, 108, 92, 58, 99, 63, 85, 104, 118, 97, 40, 20, 88, 144, 164, 39, 130, 27, 73, 22, 148, 183, 84, 138, 56, 146, 182, 172, 8, 116, 25, 32, 54, 61, 198, 194, 121, 57, 191, 37, 119, 113, 74, 125, 152, 174, 101, 19, 89, 180, 10, 96, 165, 175, 179, 13, 77, 141, 15, 80, 189, 64, 173, 110, 190, 163, 72, 14, 16, 98, 106, 49, 65, 188, 59, 29, 66, 199, 135, 36, 124, 26, 28, 151, 100, 68, 150, 128, 139, 30, 51, 127, 95, 69, 122, 4, 91, 193, 53, 75, 7, 31, 142, 109, 107, 44, 103, 140, 90, 177, 33, 60, 156, 102, 133, 168, 115, 166, 21, 134, 184, 5, 48, 46, 143, 132, 6, 169, 187, 45, 153, 3, 112, 195, 81, 167, 79, 147, 83, 11, 120, 12, 178, 76, 23, 136, 47, 82, 171, 50, 52, 24, 71, 94, 41, 34, 87, 154, 197, 9, 43, 55, 176, 114, 70, 17, 145, 160, 192, 111, 129, 155, 149, 18, 159, 93, 2, 186, 131]


In [88]:
m = dist_matrix(400)
c = repeated_greedy_tsp(m)

In [89]:
m = dist_matrix(500)
c = repeated_greedy_tsp(m)

In [21]:
def exhaustive_tsp(dist_m: np.ndarray)-> list:
    try:
        """Examina todos los posibles circuitos y devuelve aquel con la distancia más corta."""
        best_len=0
        nodes=dist_m.shape[0]
        for permutation in itertools.permutations(range(nodes)):
            perm=list(permutation)
            perm.append(perm[0])
            new_len=len_circuit(perm,dist_m)
            if(best_len>new_len or best_len==0):
                best_len=new_len
                best_perm=perm
        return best_perm
    except:
        print("Error en exhaustive_tsp")

In [35]:
print(exhaustive_tsp(dist_matrix(9)))


[0, 2, 7, 6, 8, 1, 5, 4, 3, 0]


In [34]:
print(exhaustive_tsp(dist_matrix(8)))


[0, 1, 5, 3, 6, 2, 4, 7, 0]


In [36]:
print(exhaustive_tsp(dist_matrix(10)))


[0, 2, 9, 8, 3, 1, 6, 4, 5, 7, 0]


In [38]:
print(exhaustive_tsp(dist_matrix(11)))


[0, 3, 4, 1, 2, 5, 9, 6, 8, 7, 10, 0]
